# 🌐 ViBioMIR - Kaggle High-Performance Distributed Corpus Crawler

Notebook này được thiết kế để **chạy cào dữ liệu tối ưu trên Kaggle CPU** (tiết kiệm 100% quota GPU cho khâu huấn luyện/inference).

### ✨ Đặc điểm nổi bật:
1. **Chạy hoàn toàn trên CPU (Non-GPU)**: Tận dụng đường truyền mạng tốc độ cao (1Gbps+) của Kaggle.
2. **Hỗ trợ Chia Shard linh hoạt (`--shard-id` / `--num-shards`)**: Có thể mở nhiều notebook cào song song các phần khác nhau.
3. **Domain Round-Robin & Asyncio Queue**: Không gửi dồn dập vào một tên miền để tránh bị chặn IP / 429 Too Many Requests.
4. **Tự động lưu Checkpoint (Resumable)**: Nếu mạng gián đoạn hoặc ngắt phiên, chạy lại sẽ tự động bỏ qua các link đã cào xong.
5. **Tự động làm sạch & nén Parquet ZSTD**: Lọc rác boilerplate bằng 6 Master Regex, sửa lỗi font tiếng Trung (*Mojibake*).
6. **Đồng bộ tự động lên Hugging Face Storage Bucket**: Sử dụng `hf sync` đẩy thẳng lên `hf://buckets/nieths/ViBioMIR/corpus`.

## 1. Cài đặt Môi trường & Kéo Code Mới Nhất

In [ ]:
import os

# 1. Clone repo nếu chưa có hoặc pull code mới nhất
if not os.path.exists("/kaggle/working/med-doc-retrieval"):
    !git clone https://github.com/neiths/med-doc-retrieval.git /kaggle/working/med-doc-retrieval

# 2. Di chuyển vào thư mục code
%cd /kaggle/working/med-doc-retrieval
!git pull origin main

# 3. Cài đặt các thư viện phục vụ cào dữ liệu và xử lý văn bản
!pip install -q --upgrade pip
!pip install -q trafilatura httpx beautifulsoup4 pyarrow "huggingface_hub[cli]" pyvi jieba loguru typer pydantic pyyaml tqdm
!which hf || pip install -q hf

## 2. Cấu Hình Shard Cào & Token Hugging Face

In [ ]:
# Điền Hugging Face Token có quyền Write để sync lên bucket
os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

# Cấu hình phân chia Shard (Ví dụ: chia làm 4 shards, notebook này chạy shard 0)
SHARD_ID = 0        # Thay đổi từ 0, 1, 2, 3 nếu bạn chạy nhiều máy song song
NUM_SHARDS = 4      # Tổng số shards
CONCURRENCY = 20    # Số luồng request bất đồng bộ đồng thời
OUTPUT_JSONL = f"data/raw/crawled_shard_{SHARD_ID + 1}.jsonl"

print(f"Cấu hình cào: Shard {SHARD_ID + 1}/{NUM_SHARDS} -> File: {OUTPUT_JSONL}")

## 3. Tải Danh Sách URLs Ban Tổ Chức (`links_corpus.parquet`)

In [ ]:
# Tải links_corpus.parquet (~34MB) từ repository Hugging Face của cuộc thi
urls_file = "data/raw/vibio_mir/links_corpus.parquet"

if not os.path.exists(urls_file):
    os.makedirs("data/raw/vibio_mir", exist_ok=True)
    from huggingface_hub import hf_hub_download
    print("Đang tải links_corpus.parquet từ Hugging Face AIGuruTinix/ViBioMIR...")
    hf_hub_download(
        repo_id="AIGuruTinix/ViBioMIR",
        filename="links_corpus.parquet",
        repo_type="dataset",
        local_dir="data/raw/vibio_mir",
    )
    print("Tải thành công danh sách URLs!")
else:
    print(f"Đã có sẵn danh sách URLs tại {urls_file}!")

## 4. Chạy Cào Dữ Liệu Bất Đồng Bộ (Asynchronous Crawler)

In [ ]:
# Chạy crawler với cơ chế Checkpoint Resume tự động
!python scripts/crawl_corpus.py \
    --input data/raw/vibio_mir/links_corpus.parquet \
    --output {OUTPUT_JSONL} \
    --shard-id {SHARD_ID} \
    --num-shards {NUM_SHARDS} \
    --concurrency {CONCURRENCY} \
    --min-delay 0.1 \
    --max-delay 0.4

## 5. Làm Sạch Boilerplate, Sửa Mojibake & Nén Sang Parquet ZSTD

In [ ]:
# Chuyển đổi file JSONL vừa cào thành Parquet chuẩn, làm sạch 100% bằng 6 Master Regex
!python scripts/convert_jsonl_to_parquet.py \
    --input {OUTPUT_JSONL} \
    --output-dir data/processed/parquet_corpus \
    --chunk-size 100000

## 6. Tự Động Đồng Bộ Lên Hugging Face Storage Bucket

In [ ]:
# Đẩy thẳng các file Parquet sạch lên Storage Bucket để máy khác/notebook GPU kéo về build index
!hf sync data/processed/parquet_corpus hf://buckets/nieths/ViBioMIR/corpus
print("Hoàn tất đồng bộ dữ liệu cào mới lên Hugging Face Storage Bucket!")